El objetivo de este Notebook es interpretar los resultados y demostrar para qué sirve.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
cache  data  models  notebooks	README.md  requirements.txt  results


In [ ]:
import pickle
import json
import networkx as nx

from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

# Carga de resultados

In [ ]:
with open("results/graph_multimodal.gpickle", "rb") as f:
    G = pickle.load(f)

with open("results/graph_summary.json", "r") as f:
    summary = json.load(f)

summary


{'n_nodes': 2431,
 'n_edges': 3875,
 'edges_same_ip': 3874,
 'edges_same_registrar': 0,
 'edges_with_content': 1,
 'connected_components': [49, 28, 21, 20, 18, 15, 14, 13, 13, 12]}

Se puede observar que el grafo multimodal está formado por 2431 dominios y 3875 aristas.

La mayoría de conexiones se deben a relaciones infraestructurales (IP compartida), mientras que las de contenido son escasas (se refuerza su valor como señal discriminativa).

In [ ]:
components = list(nx.connected_components(G))
component_sizes = sorted([len(c) for c in components], reverse=True)

component_sizes[:10]


[49, 28, 21, 20, 18, 15, 14, 13, 13, 12]

Compruebo los 10 componentes (clusters) más grandes del grafo y qué tamaño tienen.

En el Notebook 09, a partir de los análisis de los componentes conexos del grafo (clusters), se identificaron dos tipos de estructuras relevantes dentro del grafo:

Por un lado, se observan clusters de gran tamaño formados casi exclusivamente por relaciones infraestructurales. Un ejemplo representativo es el mayor cluster del grafo, compuesto por 49 dominios pertenecientes a la red "iheart.com". Este tipo de cluster refleja una fuerte dependencia de diferentes dominios con la misma infraestructura, típica de grandes conglomerados mediáticos.

Por otro lado, se identificó un único caso de relación basada en similitud de contenido, que conecta dos dominios distintos mediante valores extremadamente altos de similitud TF-IDF y SBERT. Este caso resulta relevante, debido a que pone de manifiesto la reutilización o duplicación de contenido entre dominios sin relación infraestructural directa.

# Ejemplo ilustrativo

Quiero seleccionar una noticia externa para demostrar que el grafo multimodal puede utilizarse para entender con qué tipo de dominios, temáticas o estructuras está más relacionada dicha noticia (sin modificar el grafo).

Noticia externa:

Fuente: BBC News

URL: https://www.bbc.com/mundo/articles/cjrz4lx4g30o

Tema: Geopolítica internacional / Estados Unidos - Venezuela

Descripción: Noticia relativa a una operación militar de Estados Unidos que culmina con la detención del presidente venezolano Nicolás Maduro y genera reacciones diplomáticas y debate internacional.

In [ ]:
# Fragmento representativo de la noticia
external_text = """
In early January 2026, United States military forces conducted an operation
that resulted in the detention of Venezuelan President Nicolás Maduro on U.S.
soil. The event triggered international responses, with discussions around
legality, diplomatic repercussions, and global geopolitical implications.
Political leaders in various countries voiced concern over the use of force
and the potential impact on regional stability.
"""

In [ ]:
# Cargo el modelo SBERT utilizado en todo el proyecto
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Generación del embedding de la noticia
external_embedding = model.encode(
    external_text,
    normalize_embeddings=True
)

In [ ]:
# Cargo las aristas SBERT entre dominios
df_content = pd.read_csv("results/content_edges_sbert.csv")

df_content.head()


,source,target,weight
0,Government News,Middle-east,0.874459
1,Government News,News,0.925310
2,Government News,US_News,0.874619
3,Government News,left-news,0.966387
4,Government News,politics,0.967762


In [ ]:
# Veo los dominios que participan en  las relaciones semánticas (SBERT)
semantic_domains = set(df_content["source"]).union(
    set(df_content["target"])
)

semantic_domains


{'Government News',
 'Middle-east',
 'News',
 'US_News',
 'left-news',
 'politics',
 'politicsNews',
 'worldnews'}

In [ ]:
# Genero los embeddings SBERT para cada dominio semántico
domain_embeddings = {
    domain: model.encode(domain, normalize_embeddings=True)
    for domain in semantic_domains
}



In [ ]:
# Defino similitud coseno
def cosine_sim(a, b):
    return np.dot(a, b)


In [ ]:
# Calculo la similitud semántica entre la noticia externa y cada dominio
scores = [
    (domain, cosine_sim(external_embedding, emb))
    for domain, emb in domain_embeddings.items()
]

# Ordeno de mayor a menor
scores = sorted(scores, key=lambda x: x[1], reverse=True)

scores


[('US_News', np.float32(0.2844031)),
 ('Government News', np.float32(0.22692534)),
 ('Middle-east', np.float32(0.20500545)),
 ('politics', np.float32(0.20170203)),
 ('politicsNews', np.float32(0.17359664)),
 ('News', np.float32(0.16010913)),
 ('worldnews', np.float32(0.13512468)),
 ('left-news', np.float32(0.09323364))]

Aunque los valores absolutos de similitud son moderados, se pueden sacar conclusiones. La noticia se asocia principalmente con los dominios "US_News", "Government News" y "Middle-east", lo que concuerda plenamente con su temática geopolítica.

Así se demuestra cómo el grafo multimodal puede utilizarse para contextualizar noticias externas sin necesidad
de modificar su estructura.